# 01 · Agent → Agent (A2A) with the auth manager

A Gemini Enterprise agent frequently needs to call **another agent** — a partner
analytics agent, a specialist sub-agent, a third-party agent published to your
org. The open **A2A (Agent2Agent) protocol** standardizes that call:

* the **remote agent** publishes an **Agent Card** (usually at
  `/.well-known/agent-card.json`) describing its skills **and its
  `securitySchemes`** (apiKey / bearer / **oauth2** / openIdConnect);
* the **client agent** reads the card, obtains a credential that satisfies the
  scheme, and attaches it to every request.

The **auth manager** is what supplies that credential — as a **2-legged**
service identity (the calling agent authenticates *as itself*) or a **3-legged**
on-behalf-of-user token (the call carries the *user's* authority).

> The *Runnable demo* below stands up a real (mock) A2A remote agent that
> enforces OAuth, and calls it through the auth manager. The *Production mapping*
> shows the ADK `RemoteA2aAgent` + `a2a` equivalent.


## Architecture

```
  Client agent (Gemini Enterprise)                 Remote A2A agent
  ┌───────────────────────────┐                    ┌──────────────────────────┐
  │  AuthManager               │  1. GET card       │  /.well-known/           │
  │   • partner-agent  (2LO)   │ ─────────────────▶ │      agent-card.json      │
  │   • user-context   (3LO)   │ ◀───────────────── │   securitySchemes: oauth2 │
  │                            │  2. token by name  │                          │
  │  headers = auth_headers()  │ ─────────────────▶ │  POST /a2a  (Bearer …)   │
  └───────────────────────────┘  3. authorized call │   → verifies token       │
                                                     └──────────────────────────┘
```


In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

from auth_provider_demo import AuthManager
from auth_provider_demo.mock_oauth_server import (
    DEMO_CLIENT_ID, DEMO_CLIENT_SECRET, MockOAuthServer,
)

RUN_PRODUCTION = False   # flip to True to run the ADK RemoteA2aAgent cell

# Mock IdP / authorization server (stands in for your Gemini Enterprise authorization).
idp = MockOAuthServer().start()

manager = AuthManager()
# 2-legged: the calling agent's own service identity.
manager.register_two_legged(
    "partner-agent",
    token_url=idp.token_url, client_id=DEMO_CLIENT_ID, client_secret=DEMO_CLIENT_SECRET,
    scope="agent.invoke",
    description="Service identity used to call the partner analytics agent",
)
# 3-legged: a per-user token, for when the remote agent must act as the user.
manager.register_three_legged(
    "user-context",
    token_url=idp.token_url, authorization_url=idp.authorize_url,
    client_id=DEMO_CLIENT_ID, client_secret=DEMO_CLIENT_SECRET,
    redirect_uri="https://app.example.com/oauth/callback",
    scope="agent.invoke.onbehalf",
    description="On-behalf-of-user authority propagated to the remote agent",
)
print("authorizations:", [a["name"] for a in manager.list_authorizations()])

authorizations: ['partner-agent', 'user-context']


## Stand up a (mock) remote A2A agent that enforces OAuth

This is a minimal but real HTTP service: it publishes an **Agent Card** declaring
an `oauth2` security scheme, and its invoke endpoint **rejects calls without a
valid bearer token**. It distinguishes a *service* caller (2-legged token) from
an *on-behalf-of-user* caller (3-legged token) — just like a real resource
server authorizing by token claims.

In [2]:
import json, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

def _principal_for(token: str):
    """Toy token validation: real servers verify signature/introspection."""
    if token.startswith("app-access-"):
        return {"type": "service", "id": "partner-agent-client"}
    if token.startswith("user-access-"):
        return {"type": "user", "id": "propagated-end-user"}
    return None

class RemoteAgentHandler(BaseHTTPRequestHandler):
    def log_message(self, *a):  # keep notebook output clean
        return

    def _send(self, code, payload):
        body = json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path == "/.well-known/agent-card.json":
            self._send(200, AGENT_CARD)
        else:
            self._send(404, {"error": "not_found"})

    def do_POST(self):
        if self.path != "/a2a":
            return self._send(404, {"error": "not_found"})
        auth = self.headers.get("Authorization", "")
        token = auth[7:] if auth.startswith("Bearer ") else ""
        principal = _principal_for(token)
        if principal is None:
            # A2A/OAuth: signal how to authenticate.
            self.send_response(401)
            self.send_header("WWW-Authenticate", 'Bearer realm="a2a"')
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(json.dumps({"error": "invalid_token"}).encode())
            return
        self._send(200, {
            "result": "3 critical findings in scope",
            "authorized_as": principal,
        })

_srv = ThreadingHTTPServer(("127.0.0.1", 0), RemoteAgentHandler)
REMOTE_BASE = f"http://127.0.0.1:{_srv.server_address[1]}"

AGENT_CARD = {
    "name": "analytics-agent",
    "description": "Returns security analytics for the caller.",
    "url": f"{REMOTE_BASE}/a2a",
    "version": "1.0.0",
    "capabilities": {"streaming": False},
    "defaultInputModes": ["text"], "defaultOutputModes": ["text"],
    "securitySchemes": {
        "oauth2": {
            "type": "oauth2",
            "flows": {"clientCredentials": {
                "tokenUrl": idp.token_url,
                "scopes": {"agent.invoke": "Invoke the analytics agent"},
            }},
        }
    },
    "security": [{"oauth2": ["agent.invoke"]}],
    "skills": [{"id": "critical-findings", "name": "Critical findings",
                "description": "List critical findings", "tags": ["security"]}],
}

threading.Thread(target=_srv.serve_forever, daemon=True).start()
print("remote A2A agent listening at", REMOTE_BASE)

remote A2A agent listening at http://127.0.0.1:37015


### 1. Discover the Agent Card

The client agent fetches the card and reads `securitySchemes` to learn *how* to
authenticate before making any call.

In [3]:
import urllib.request

def http_json(method, url, headers=None, data=None):
    req = urllib.request.Request(url, method=method, headers=headers or {})
    try:
        with urllib.request.urlopen(req, data=data) as r:
            return r.status, json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read().decode())

import urllib.error
_, card = http_json("GET", f"{REMOTE_BASE}/.well-known/agent-card.json")
print("skills          :", [s["name"] for s in card["skills"]])
print("security schemes:", list(card["securitySchemes"]))
print("required scopes :", card["security"])

skills          : ['Critical findings']
security schemes: ['oauth2']
required scopes : [{'oauth2': ['agent.invoke']}]


### 2. Unauthenticated call is rejected

Without a bearer token the remote agent returns **401** and a
`WWW-Authenticate` challenge.

In [4]:
status, body = http_json("POST", card["url"], headers={"Content-Type": "application/json"},
                         data=json.dumps({"message": "list critical findings"}).encode())
print("status:", status, "->", body)

status: 401 -> {'error': 'invalid_token'}


### 3. Call with a **2-legged** service identity

The auth manager mints the calling agent's own token for the `partner-agent`
authorization. The remote agent authorizes it as a **service** principal.

In [5]:
headers = {"Content-Type": "application/json"}
headers.update(manager.authorization_headers("partner-agent"))   # <-- auth manager
status, body = http_json("POST", card["url"], headers=headers,
                         data=json.dumps({"message": "list critical findings"}).encode())
print("status:", status)
print("result:", body["result"])
print("authorized_as:", body["authorized_as"])

status: 200
result: 3 critical findings in scope
authorized_as: {'type': 'service', 'id': 'partner-agent-client'}


### 4. Call **on behalf of a user** (3-legged)

When the remote agent must act with the *user's* authority (e.g. it will read
that user's resources downstream), the client propagates a per-user token
instead. The user consents once; the manager then issues user tokens.

In [6]:
import urllib.request, urllib.error

def simulate_user_consent(consent_url):
    class _NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, *a, **k): return None
    opener = urllib.request.build_opener(_NoRedirect)
    try:
        opener.open(consent_url); raise AssertionError("expected redirect")
    except urllib.error.HTTPError as e:
        return e.headers["Location"]

user = "alice"
if manager.needs_consent("user-context", user_id=user):
    manager.complete_consent("user-context", simulate_user_consent(
        manager.consent_url("user-context", user_id=user)))

headers = {"Content-Type": "application/json"}
headers.update(manager.authorization_headers("user-context", user_id=user))  # <-- per-user
status, body = http_json("POST", card["url"], headers=headers,
                         data=json.dumps({"message": "list my critical findings"}).encode())
print("status:", status)
print("authorized_as:", body["authorized_as"])

status: 200
authorized_as: {'type': 'user', 'id': 'propagated-end-user'}


Same client code path, same auth-manager API — only the **authorization name**
(and `user_id`) changed to switch between *service identity* and
*on-behalf-of-user*.


## Production mapping — ADK `RemoteA2aAgent` + `a2a`

In ADK you wrap a remote agent as a `RemoteA2aAgent` pointed at its Agent Card.
The card's `securitySchemes` drive credential selection; you supply the token
from your credential service / Gemini Enterprise authorization.

```python
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.agents import LlmAgent

analytics = RemoteA2aAgent(
    name="analytics_agent",
    description="Partner security-analytics agent.",
    agent_card=f"{REMOTE_BASE}/.well-known/agent-card.json",
)

root_agent = LlmAgent(
    model="gemini-2.0-flash",
    name="orchestrator",
    instruction="Delegate analytics questions to analytics_agent.",
    sub_agents=[analytics],
)
```

Defining the *server* side card with the `a2a` SDK (note the `oauth2`
`security_schemes` — the same shape the mock served):

```python
from a2a.types import (
    AgentCard, AgentCapabilities, AgentSkill,
    OAuth2SecurityScheme, OAuthFlows, ClientCredentialsOAuthFlow,
)

card = AgentCard(
    name="analytics-agent",
    description="Returns security analytics for the caller.",
    url="https://analytics.internal/a2a",
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=False),
    default_input_modes=["text"], default_output_modes=["text"],
    security_schemes={
        "oauth2": OAuth2SecurityScheme(flows=OAuthFlows(
            client_credentials=ClientCredentialsOAuthFlow(
                token_url="https://oauth2.googleapis.com/token",
                scopes={"agent.invoke": "Invoke the analytics agent"},
            )))
    },
    security=[{"oauth2": ["agent.invoke"]}],
    skills=[AgentSkill(id="critical-findings", name="Critical findings",
                       description="List critical findings", tags=["security"])],
)
```

**In Gemini Enterprise**, register the partner agent and attach the
`partner-agent` (2-legged) or `user-context` (3-legged) **Authorization** from
notebook 00; the platform injects the token when the orchestrator calls it.


In [7]:
if RUN_PRODUCTION:
    from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
    remote = RemoteA2aAgent(
        name="analytics_agent",
        description="Partner security-analytics agent.",
        agent_card=f"{REMOTE_BASE}/.well-known/agent-card.json",
    )
    print("Constructed RemoteA2aAgent:", remote.name)
else:
    print("RUN_PRODUCTION is False — skipping ADK RemoteA2aAgent construction.")

RUN_PRODUCTION is False — skipping ADK RemoteA2aAgent construction.


## Summary

* A2A auth is declared by the remote agent's **Agent Card** `securitySchemes`.
* The **auth manager** supplies the matching credential by name — **2-legged**
  for a service identity, **3-legged** to propagate a user's authority.
* The client code is identical across flows; only the authorization name changes.

Next: **02_agent_to_mcp_auth.ipynb** — the same manager, applied to MCP tools.


In [8]:
_srv.shutdown(); _srv.server_close(); idp.stop()
print("servers stopped.")

servers stopped.
